In [ ]:
!pip install seaborn matplotlib
#import seaborn as sns
#sns.__version__

In [ ]:
!pip install -q pyALE==1.2.0
!pip install -U shap -q


## **Взаимодействие признаков на мандаринках**

Всем привет! Эта практика была подготовлена в Новый год. В ней мы будем работать с датасетом о мандаринках и посмотрим на признаки этого чудного цитруса со всех сторон!

P.S. Надеемся, аллергии на мандарины у вас нет!
Приятного кодинга!

### **Постановка задачи**
Рассмотрим решение задачи прогнозирования качества мандаринов. С точки зрения постановки, качество есть непрерывная переменная в диапазоне от 0 до 5. Промежуточные значения: `1.5, ..., 4.5` также значимы, а представленные в датасете точки оценки, хоть и дискретны с шагом 0.5, но несбалансированы. Поэтому пробуем решить задачу регрессии.

In [ ]:
import pandas as pd
import numpy as np
from PyALE import ale

import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay, partial_dependence

from shap import TreeExplainer

from sklearn.tree import plot_tree


LINK = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/orange_dataset.csv'
data = pd.read_csv(LINK, index_col=0)

### **Построение модели**

В ячейке ниже мы просто обучим модель, используя `MinMaxScaler`. Преобразование (масштабирование) осуществляется так:

1. Определяем желаемый диапазон признаков — обычно $(0, 1)$. Обозначим $x_{min} = 0, x_{max} = 1$:

2. Масштабируем данные по формуле:

$$X_{scaled} = \frac{X-\min X}{\max X - \min X}\,(x_{max} - x_{min}) + x_{min}$$

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Подготовка датасета к обучению
X = data.drop('Quality (1-5)', axis=1)
y = data['Quality (1-5)']

feature_names = list(data.columns)
feature_names.remove('Quality (1-5)')

# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Масштабирование
sc = MinMaxScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

X_train = pd.DataFrame(X_train, columns = feature_names)
X_test = pd.DataFrame(X_test, columns = feature_names)

В качестве модели будем использовать решающее дерево.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Обучение модели
tree = DecisionTreeRegressor(random_state=42, max_depth=4)
tree.fit(X_train, y_train)

#predictions = np.clip(tree.predict(X_test), 1, 5)
predictions = tree.predict(X_test)

mse = mean_squared_error(y_test, predictions)
print(f'Model MSE : {mse}, rmse: {mse**(1/2)}')

**Примечание:** Мы могли бы сжимать прогноз, чтобы не выходить за диапазоны. Но дерево, по построению, не будет выходить за диапазон, на котором оно было обучено.

**Задание 1.**

Сделайте прогноз на тестовых данных `X_test`. Какое максимальное значение прогноза?

Ответ округлите до сотых.

In [ ]:
# Ваш код здесь

### **Построение карт значимости**

Давайте выберем пару признаков для анализа, исходя из их смысла в задаче. Среди признаков, у нас есть:

- `Size (cm)` — размер мандарина в сантиметрах.  
- `Weight (g)` — вес мандарина в граммах.  
- `Brix (Sweetness)` — показатель сладости мандарина
- `pH (Acidity)` — уровень кислотности мандарина.  
- `Softness (1-5)` — мягкость мандарина по шкале от 1 (очень твердый) до 5 (очень мягкий).  
- `HarvestTime (days)` — время сбора урожая, указанное в днях с момента начала сезона.  
- `Ripeness (1-5)` — степень зрелости мандарина по шкале от 1 (незрелый) до 5 (полностью зрелый).  
- `Blemishes (Y/N)` — наличие повреждений или дефектов на кожуре мандарина (Y — есть, N — нет).  
- `Color_Light Orange` — цвет мандарина: светло-оранжевый.  
- `Color_Orange` — цвет мандарина: оранжевый.  
- `Color_Orange-Red` — цвет мандарина: оранжево-красный.  
- `Color_Yellow-Orange` — цвет мандарина: желто-оранжевый.  

**Задание 2.**

Сколько в данных бинарных признаков?

`Ваш ответ здесь.`

### **Сладость и кислотность**
Выберем признаки для анализа. Давайте посмотрим, как влияют сладость `Brix (Sweetness)` и кислотность `pH (Acidity)` на прогноз. Для начала посмотрим одномерное влияние при помощи **PDP** и **ALE** графиков.

In [ ]:
feature_idx = 2
feature_name = data.columns[feature_idx]

print(f'Анализируем {feature_name}')

In [ ]:
results = partial_dependence(tree, X_test, [feature_idx], grid_resolution=20, kind='average')
#results = partial_dependence(tree, X_test, [0], grid_resolution=20, kind='individual') == ICE

results
plt.plot(results['grid_values'][0], results['average'][0])

plt.title(f'PDP plot for {feature_name}')
plt.xlabel(f'{feature_name} value')
plt.ylabel('model prediction');

Видно, что есть некоторый порог сладости, после которого модель склонна присваивать мандарину лучшее качество. А что до кислотности?

**Задание 3.**
Постройте одномерный график для кислотности и сделайте вывод о характере влияния.


In [ ]:
feature_idx = # Ваш код здесь
feature_name = # Ваш код здесь

print(f'Анализируем {feature_name}')

In [ ]:
# Ваш код здесь

results =

plt.plot(results['grid_values'][0], results['average'][0])

plt.title(f'PDP plot for {feature_name}')
plt.xlabel(f'{feature_name} value')
plt.ylabel('model prediction');

Если мы рассмотрим ALE, то зависимость будет похожей. Можете убедиться в этом, построив ALE для ph сами.

In [ ]:
brix = 'Brix (Sweetness)'
ph = 'pH (Acidity)'

ale_eff = ale(
    X=pd.DataFrame(X_test, columns=feature_names), model=tree, feature=[brix], grid_size=50, include_CI=True
)

In [ ]:
# Ваш код здесь

### **Оценка двумерного влияния**

Перейдем к оценке признаков в паре. Посмотрим на совместное влияние `Brix (Sweetness)` и `pH (Acidity)` двумя способами.

In [ ]:
# Построим pdp график
PartialDependenceDisplay.from_estimator(tree, X_test, [(2, 3)])
plt.title('PDP plot for interactions');

**Задание 4.**
Проанализируйте в тетради двумерный PDP-график для `Brix` и `pH`, построенный перед заданием 4. Аналогично, постройте двумерный график для `Brix` в паре с признаками `Size, Weight, Harvest Time`.

Гипотезу о совместном влиянии `Brix` с каким из признаков можно выдвинуть в первую очередь — у какой пары срезы карты по `Brix` при разных значениях второго признака сильнее меняют форму, а не только сдвигаются?

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`

**Задание 5.**

Для признака из задания 4 постройте ALE. Что происходит с совместным эффектом, если поправить его на интервалы?

In [ ]:
# подсказка: используйте impute_empty_cells

# Ваш код здесь


### **SHAP**

Получение interaction values в shap повторяет шаги построения данного метода анализа вклада признаков, для случая, когда мы вычисляем индивидуальный вклад.

In [ ]:
pred = tree.predict(X_test)

explainer = TreeExplainer(tree)
explanation = explainer(X_test)

**Задание 6.**

С помощью атрибутов explainer вытащите interaction values. Вам поможет `dir(explainer)`. Атрибут в названии содержит слово `interaction`. Сколько размерностей у полученного объекта?

In [ ]:
# Ваш код здесь

Сравните это с размерностью `X_test`. Парное взаимодействие содержится для каждого объекта в данных.

Проанализируем парное взаимодействие для одного объекта. Зафиксируем объект с индексом 0 и посмотрим парные эффекты для него.

In [ ]:
pd.DataFrame(shap_interaction[0],index=feature_names,columns=feature_names)

**Задание 7.** Вместе с признаком какого типа (кроме самого себя) `Brix` оказывает наиболее сильное взаимодействие для объекта с индексом 0?

`Ваш ответ здесь`

In [ ]:
# Базовое значение — explainer.expected_value: средний прогноз дерева по обучающим данным,
# прошедшим через его листья, а не средний прогноз на тестовой выборке
base_value = np.ravel(explainer.expected_value)[0]
mean_pred = np.mean(pred)

sum_shap = np.sum(shap_interaction[0])

print(f"Прогноз модели: {pred[0]}")
print(f"Базовое значение: {base_value}, сумма shap: {sum_shap}, базовое значение + сумма shap: {base_value + sum_shap}")
print(f"Средний прогноз на тестовой выборке: {mean_pred}")

Вот так: базовое значение `explainer.expected_value` плюс сумма всех взаимодействий объекта дает в точности прогноз модели. Базовое значение — средний прогноз дерева на обучающих данных; средний прогноз на тестовой выборке от него отличается, и с ним равенство не выполняется.

Более информативно анализировать взаимодействия для всех сразу, в виде матрицы. Для этого достаточно усреднить полученный выход по объектам (нулевой оси) и построить тепловую карту.

In [ ]:
import seaborn as sns
# Усреднение (можно взять медиану — она более устойчива к "слишком большим" или "слишком маленьким" среди всех значений — на всякий случай)
mean_shap = np.round(np.median(np.abs(shap_interaction), 0), 4)

df = pd.DataFrame(mean_shap,index=feature_names,columns=feature_names)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(df, ax=ax, annot=df)

plt.title('SHAP heatmap')
plt.xticks(rotation=45);

**Задание 8.** Вместе с каким признаком (кроме самого себя) `Brix` взаимодействует больше всего в среднем по датасету?

`Ваш ответ здесь.`

**Задание 9.**

Проверьте, согласуется ли пара, на которую указали карты PDP в задании 4, с первой парой у SHAP-взаимодействий?

`Ваш ответ здесь.`

### **H-statistic**

Наконец, используем H-статистику. Проверьте, согласуется ли результат с shap?

In [ ]:
from itertools import combinations
from sklearn.inspection import partial_dependence

def centered_pd(model, X, features):
    """Центрированная PD-функция в точках данных, как у Friedman и Popescu (2008).

    Для каждого объекта i признаки features всех объектов выборки заменяются его значениями,
    прогноз усредняется по выборке; из значений вычтено их среднее по объектам.
    На n объектах это n^2 прогнозов — на 49 объектах тестовой части 2401.
    """
    n = len(X)
    X_rep = pd.concat([X] * n, ignore_index=True)                  # n копий выборки подряд
    X_rep[features] = np.repeat(X[features].values, n, axis=0)    # в копии i — значения объекта i
    values = model.predict(X_rep).reshape(n, n).mean(axis=1)
    return values - values.mean()

def h_statistic(model, X, feat_i, feat_j, eps=1e-3):
    """H-статистика Фридмана для пары признаков — прямо по формуле из урока.

    H^2 = sum_i [PD_kj(x_i) - PD_k(x_i) - PD_j(x_i)]^2 / sum_i PD_kj(x_i)^2,

    суммы — по объектам выборки, все PD-функции центрированы, иначе разность не обращается в ноль.
    Если знаменатель равен нулю или близок к нему (совместного эффекта у пары нет),
    отношение не определено или неустойчиво. Такие пары помечаем NaN.
    """
    pd_ij = centered_pd(model, X, [feat_i, feat_j])
    pd_i = centered_pd(model, X, [feat_i])
    pd_j = centered_pd(model, X, [feat_j])

    numerator = ((pd_ij - pd_i - pd_j) ** 2).sum()
    denominator = (pd_ij ** 2).sum()
    if denominator < eps:            # пара не влияет на прогноз
        return float("nan")
    return numerator / denominator

# считаем H для всех пар признаков
pairs = list(combinations(X_test.columns, 2))
h_values = {f"{a} — {b}": h_statistic(tree, X_test, a, b) for a, b in pairs}
h_series = pd.Series(h_values).dropna().sort_values(ascending=False)

print(f"пар всего: {len(pairs)}, с определенной H: {len(h_series)}")
h_series.head(10)


**Задания урока «Закрывающие задания: H-статистика и выбор метода».** Ответьте по выводу ячейки выше, числа округлите до сотых.

1. Для скольких пар признаков из 66 значение $H^2$ определено, то есть не помечено NaN?
2. Чему равно наибольшее значение $H^2$ среди пар?
3. Чему равно $H^2$ пары `Brix (Sweetness) — Softness (1-5)`? Сравните ее место по $H^2$ с ее местом по мере Гринвелла в следующей секции.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
h_series.head(12).plot.barh(ax=ax, color="#2a78d6")
ax.invert_yaxis()
ax.set_xlabel("$H^2$")
ax.set_title("H-статистика Фридмана по парам признаков")
plt.tight_layout();


Заметьте, что H-статистика предлагает больше взаимодействий, чем shap. Насколько это правда? Проверять необходимо в модели. На практике значения Шепли более устойчивы.

### **Мера Гринвелла (interaction strength)**

H-статистика дорога: для одной пары нужно $O(N^2)$ обращений к модели. Гринвелл предложил меру,
которая считается по одной двумерной PD-сетке — без вложенного перебора по объектам.

Идея простая. Если два признака не взаимодействуют, кривая частичной зависимости первого
выглядит одинаково при любом значении второго: форма не меняется, кривая только сдвигается по
вертикали. Если взаимодействие есть, форма плывет. Осталось измерить, насколько сильно:

$$\mathcal{I}(x_i, x_j) = \frac{1}{2}\Big[\ \mathrm{sd}_{j}\big(\mathrm{sd}_{i}\,\widehat{PD}(x_i,x_j)\big) \ +\ \mathrm{sd}_{i}\big(\mathrm{sd}_{j}\,\widehat{PD}(x_i,x_j)\big)\Big]$$

Внутреннее стандартное отклонение берется по одной переменной при фиксированной другой,
внешнее — по получившемуся набору. Множитель $\tfrac12$ симметризует величину.

**Где это реализовано.** Авторская реализация была в R-пакете
[`vip`](https://cran.r-project.org/package=vip), функция `vint()`: пакет написал сам
Гринвелл. В версии vip 0.4.0 функцию удалили; в R мера теперь есть в пакете
[`moreparty`](https://cran.r-project.org/package=moreparty) (`GetInteractionStrength()`, для моделей `cforest`).
В Python — [`artemis`](https://github.com/pyartemis/artemis) (на PyPI `pyartemis`, `GreenwellMethod`). Еще один близкий прием —
[`scikit-explain`](https://scikit-explain.readthedocs.io/en/latest/), вызов
`ale_variance(..., interaction=True)`: прием тот же, только разброс считается по ALE, а не по PD.
Осторожно с названиями: в том же `scikit-explain` есть функция `interaction_strength()`, но она
считает другую величину — меру IAS из работы Molnar et al., 2020.

**Сколько это стоит.** Ничего заметного: одна 2D-PD-сетка на пару, на наших данных — доли
секунды. Ради этого меру и считают перед H-статистикой, чтобы сузить список пар.

In [ ]:
def interaction_strength(model, X, feat_i, feat_j, grid_resolution=20):
    """Мера Гринвелла для пары признаков — прямо по формуле выше.

    Считается по одной двумерной PD-сетке: внутреннее sd берется по одной оси
    при фиксированной другой, внешнее — по получившемуся набору. Симметризуем,
    усредняя два направления.
    """
    cols = list(X.columns)
    i, j = cols.index(feat_i), cols.index(feat_j)

    pd_res = partial_dependence(model, X, features=[(i, j)],
                                grid_resolution=grid_resolution, kind="average")
    surface = pd_res["average"][0]                 # 2D-PD на сетке: (grid_i, grid_j)

    inner_i = surface.std(axis=0, ddof=1)          # разброс по i при каждом фиксированном j
    inner_j = surface.std(axis=1, ddof=1)          # и симметрично
    return 0.5 * (inner_i.std(ddof=1) + inner_j.std(ddof=1))


# считаем меру для тех же пар, что и H-статистику
g_values = {f"{a} — {b}": interaction_strength(tree, X_test, a, b) for a, b in pairs}
g_series = pd.Series(g_values).sort_values(ascending=False)

print(f"пар всего: {len(pairs)}")
g_series.head(10)

**Задание 10.** Посчитайте меру Гринвелла для всех пар признаков (ячейка выше). Какая пара оказалась на первом месте?

`Ваш ответ здесь.`

**Задание 11.** Чему равна мера Гринвелла для пары-лидера? Ответ округлите до сотых.

`Ваш ответ здесь.`

Обратите внимание на шкалу. У $H^2$ она есть: это доля дисперсии совместного эффекта,
не объяснимая суммой главных, — безразмерное число, от 0 до 1 при независимых признаках; между моделями его сравнивают только вместе с нулевым распределением. У меры Гринвелла шкалы
нет: она измеряется в единицах прогноза, и «много» или «мало» видно только при сравнении пар между
собой. Поэтому ее читают как порядок, а не как значение.

In [ ]:
# две меры рядом: ранг по каждой
both = pd.DataFrame({"H^2": h_series, "Greenwell": g_series}).dropna()
both["ранг H"] = both["H^2"].rank(ascending=False).astype(int)
both["ранг Greenwell"] = both["Greenwell"].rank(ascending=False).astype(int)
both.sort_values("ранг H").head(12)

**Задание на сравнение рангов.**

Сравните два столбца рангов. Совпадает ли первая тройка пар у обеих мер? Назовите пару, у которой
ранги расходятся сильнее всего, и объясните, почему это возможно: обе меры считаются по одной и
той же PD-функции, но измеряют разное — доля необъясненной дисперсии против изменчивости формы
кривой.

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`

**Замечание про модель.** Мы считаем обе меры на решающем дереве. Дерево дает ступенчатую
PD-поверхность, и на ней разброс формы смазан — ранжирование пар получается менее контрастным,
чем на гладкой модели. Перезапустите тетрадь со случайным лесом и посмотрите, разъезжаются ли
ранги сильнее или, наоборот, сходятся.

В качестве упражнения — посмотрите на код ниже и структуру дерева в нем. Полученные нами результаты во многом обусловлены структурой модели и их было бы быстрее найти, вытащив структуру дерева. Однако, в случае более сложных алгоритмов, графики помогают обойтись без разбора сложной структуры. Просто попробуйте перезапустить тетрадь с Random Forest.

In [ ]:
fig, ax = plt.subplots(figsize=(25, 10))
plot_tree(tree, proportion=True, ax=ax, feature_names=feature_names);

### **IAS: сколько в модели неаддитивности**

В уроке «IAS и меры вне ANOVA» разобран IAS — одно число на всю модель. Строим аддитивное приближение модели: средний прогноз плюс одномерные ALE-кривые всех признаков,

$$f_1(x) = \bar f + \sum_j g_j(x_j),$$

и смотрим, какую долю разброса прогноза вокруг среднего оно не воспроизводит:

$$\mathrm{IAS} = \frac{\sum_i \big(f(x_i) - f_1(x_i)\big)^2}{\sum_i \big(f(x_i) - \bar f\big)^2}.$$

Функция ниже считает ALE-кривую признака пакетом `PyALE` на тестовой части (49 объектов). Значение кривой в точке объекта берется линейной интерполяцией между узлами сетки, и кривая центрируется так, чтобы ее среднее по объектам было нулем.

In [ ]:
import logging
logging.getLogger('PyALE').setLevel(logging.WARNING)


def ale_curve(model, X, feat, grid_size=20):
    """Одномерная ALE-кривая признака feat в точках объектов X, центрированная по объектам."""
    eff = ale(X=X, model=model, feature=[feat], grid_size=grid_size, include_CI=False, plot=False)['eff']
    g = np.interp(X[feat].values, eff.index.values.astype(float), eff.values)
    return g - g.mean()

**Задание 12.** Посчитайте IAS решающего дерева на тестовой части, взяв ALE-кривые всех двенадцати признаков функцией `ale_curve` с `grid_size=20`. Ответ округлите до сотых.

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`

IAS зависит от сетки ALE-кривых: на тех же 49 объектах при `grid_size=10` и `grid_size=30` он получается 0.27 и 0.28. Поэтому вместе со значением IAS называют сетку, на которой он посчитан.

### **Перестановочное взаимодействие: сколько нужно повторов**

В уроке «IAS и меры вне ANOVA» разобрана мера, которой нужны истинные ответы $y$:

$$\mathrm{Interact}(i,j) = \mathrm{Err}(\{i,j\}) - \big[\mathrm{Err}(i) + \mathrm{Err}(j)\big],$$

где $\mathrm{Err}(S)$ — прирост MSE модели, когда признаки из множества $S$ перемешаны. Перестановка случайна, поэтому одно значение меры ничего не измеряет.

Ниже мера для пары `Brix` — `Softness` считается на тестовой части (49 объектов) в 20 запусках с разными сидами, и в каждом запуске усредняется по 1, 10, 100 и 1000 повторам перестановки. Каждый признак перемешивается своей перестановкой. Посмотрите, как меняется размах между запусками и остается ли знак одним и тем же: об этом задание 13 после ячейки.

In [ ]:
from sklearn.metrics import mean_squared_error


def perm_error(model, X, y, cols, rng):
    """Прирост MSE, когда признаки из cols перемешаны, каждый своей перестановкой."""
    X_perm = X.copy()
    for col in cols:
        X_perm[col] = rng.permutation(X_perm[col].values)
    return mean_squared_error(y, model.predict(X_perm)) - mean_squared_error(y, model.predict(X))


def perm_interaction(model, X, y, feat_i, feat_j, n_repeats, rng):
    """Interact(i, j) = Err({i, j}) - [Err(i) + Err(j)], среднее по n_repeats перестановкам."""
    values = [perm_error(model, X, y, [feat_i, feat_j], rng)
              - perm_error(model, X, y, [feat_i], rng)
              - perm_error(model, X, y, [feat_j], rng) for _ in range(n_repeats)]
    return np.mean(values)


softness = 'Softness (1-5)'
for n_repeats in (1, 10, 100, 1000):
    runs = [perm_interaction(tree, X_test, y_test.values, brix, softness, n_repeats, np.random.default_rng(seed))
            for seed in range(20)]
    print(f'повторов {n_repeats:4d}: 20 запусков от {min(runs):+.3f} до {max(runs):+.3f}, '
          f'размах {max(runs) - min(runs):.3f}, среднее {np.mean(runs):+.3f}')

**Задание 13.** По выводу ячейки выше: при каких числах повторов перестановки (из 1, 10, 100 и 1000) знак меры для пары `Brix` — `Softness` один и тот же во всех 20 запусках?

`Ваш ответ здесь.`

### **MEC: насколько сложны главные эффекты**

В том же уроке разобрана MEC — мера сложности главных эффектов; к взаимодействиям она не относится. Каждую одномерную ALE-кривую приближают ломаной из как можно меньшего числа отрезков $K$, пока $R^2$ приближения не станет не меньше $0.95$ (не больше 10 отрезков). Потом наклоны отрезков по одному обнуляют, пока $R^2$ остается не ниже $0.95$, и записывают число параметров ломаной $m_j = K + s_j - 1$, где $s_j$ — число оставшихся ненулевых наклонов (Molnar, Casalicchio и Bischl, 2020, алгоритм 2). Общая мера — среднее по признакам, взвешенное дисперсией кривой по объектам $v_j$:

$$\mathrm{MEC} = \frac{\sum_j v_j m_j}{\sum_j v_j}.$$

Точки излома ниже ищутся так же, как в `scikit-explain`: они добавляются по одной среди узлов сетки ALE-кривой, каждый раз та, что сильнее всего поднимает $R^2$, и на каждом отрезке подбирается своя прямая. Обнуления наклонов в `scikit-explain` нет: его `main_effect_complexity()` возвращает число отрезков $K$, а не число параметров $m_j$. У ступенчатых кривых решающего дерева часть отрезков обходится без наклона, поэтому $m_j$ может отличаться от $K$. Признак с постоянной кривой получает нулевой вес.

In [ ]:
def ale_points(model, X, feat, grid_size=20):
    """Узлы сетки и значения ALE-кривой признака feat."""
    eff = ale(X=X, model=model, feature=[feat], grid_size=grid_size, include_CI=False, plot=False)['eff']
    return eff.index.values.astype(float), eff.values


def r2(curve, approx):
    return 1 - ((curve - approx) ** 2).sum() / ((curve - curve.mean()) ** 2).sum()


def piecewise_fit(x, curve, breaks, flat=frozenset()):
    """Ломаная: на каждом отрезке между точками излома — своя прямая по МНК; у отрезков из flat наклон обнулен."""
    bounds = [0] + breaks + [len(x) - 1]
    approx = np.empty_like(curve)
    for k, (a, b) in enumerate(zip(bounds[:-1], bounds[1:])):
        if k in flat:
            approx[a:b + 1] = curve[a:b + 1].mean()
        else:
            slope, intercept = np.polyfit(x[a:b + 1], curve[a:b + 1], 1)
            approx[a:b + 1] = slope * x[a:b + 1] + intercept
    return approx


def find_breaks(x, curve, approx_error=0.05, max_segments=10):
    """Точки излома по одной, каждый раз та, что сильнее всего поднимает R^2, пока R^2 не станет не меньше 1 - approx_error."""
    breaks = []
    while (len(breaks) + 1 < max_segments and len(breaks) < len(x) - 2
           and r2(curve, piecewise_fit(x, curve, breaks)) < 1 - approx_error):
        candidates = [b for b in range(1, len(x) - 1) if b not in breaks]
        best = max(candidates, key=lambda b: r2(curve, piecewise_fit(x, curve, sorted(breaks + [b]))))
        breaks = sorted(breaks + [best])
    return breaks


def mec_params(x, curve, approx_error=0.05, max_segments=10):
    """Число отрезков K и ненулевых наклонов s; m_j = K + s - 1 (Molnar, Casalicchio и Bischl, 2020, алгоритм 2)."""
    if np.ptp(curve) == 0:
        return 1, 0
    breaks = find_breaks(x, curve, approx_error, max_segments)
    K = len(breaks) + 1
    flat = frozenset()
    while len(flat) < K:
        best = max((k for k in range(K) if k not in flat), key=lambda k: r2(curve, piecewise_fit(x, curve, breaks, flat | {k})))
        if r2(curve, piecewise_fit(x, curve, breaks, flat | {best})) < 1 - approx_error:
            break
        flat = flat | {best}
    return K, K - len(flat)

**Задание 14.** Посчитайте MEC решающего дерева на тестовой части с `grid_size=20`: число параметров $m_j = K + s_j - 1$ — функцией `mec_params` по узлам `ale_points`, вес $v_j$ — дисперсия кривой `ale_curve` по объектам. Ответ округлите до десятых.

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`